# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring a dataset using the `mlcroissant` library, referencing all entities by their `@id` fields as defined in the Croissant schema.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset as a mlcroissant.Dataset object
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes directly
print(f"Loaded: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Keywords: {', '.join(dataset.metadata.keywords)}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data into *record sets*. Each record set contains a collection of *fields* (columns). We will display all available record sets and for each, print its associated fields using their `@id` values.

In [ ]:
# List all available record sets and their fields by @id

print("Record sets in the dataset:")
record_sets = dataset.metadata.record_sets
for record_set in record_sets:
    print(f"\nRecord Set Name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    if hasattr(record_set, 'description'):
        print(f"  Description: {record_set.description}")
    fields = record_set.fields
    print("  Fields:")
    for field in fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We use the `@id` for referencing record sets. Replace variables as appropriate based on the overview above.

For this section, we'll select the first record set as our example. You can inspect or extract from any other record set by substituting its `@id`.

In [ ]:
# Extract data from all record sets into DataFrames

# Collect all record set @id values
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df

# Display available DataFrames and columns for the first record set
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(f"Record set @id being displayed: {first_record_set_id}\nColumns available:")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing, and grouping using field `@id`s from the selected record set.

*Note*: For demonstration, we'll use a typical numeric field such as 'Coverage(%)' or 'MW' if present; adapt the `numeric_field_id` as necessary to explore other columns.

In [ ]:
import numpy as np

# --- Select record set and fields by @id ---

# Use the first record set for demonstration
record_set_id = record_set_ids[0] if record_set_ids else None

if record_set_id:
    df = dataframes[record_set_id]
    # List all numeric-looking fields
    numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    if not numeric_fields:
        # Try to infer where number columns might have string dtype (CSV import)
        sample_row = df.iloc[0]
        likely_numeric = [col for col in df.columns if any(char.isdigit() for char in str(sample_row[col]))]
        print(f"Likely numeric fields detected: {likely_numeric}")
        # For this demonstration, hard-code an example field that is typical for proteomics datasets
        # e.g. 'Coverage(%)', 'MW', 'NumPeptides'; Update if field names differ in the dataset
        candidate_fields = [col for col in df.columns if 'coverage' in col.lower() or 'mw' in col.lower() or 'num' in col.lower()]
        print(f"Candidate numeric fields for EDA: {candidate_fields}")
        numeric_field_id = candidate_fields[0] if candidate_fields else likely_numeric[0] if likely_numeric else df.columns[0]
        # Try converting to numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    else:
        numeric_field_id = numeric_fields[0]

    print(f"Analyzing numeric field: {numeric_field_id} (referenced by column label)")

    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    
    # Filter records by threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean value):")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field if available
    # Pick a common field for proteomics: e.g. 'Description', 'Gene', etc.
    group_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
    group_field = group_fields[0] if group_fields else None
    if group_field:
        print(f"Grouping by categorical field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        display(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No data to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot the distribution of the selected numeric field, and, if a categorical group is available, display its aggregated mean as a horizontal bar chart.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping is available, plot aggregate
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10,5))
        display_df = grouped_df.copy()
        # Show top 10 groups by mean value
        display_df = display_df.sort_values(numeric_field_id, ascending=False).head(10)
        sns.barplot(data=display_df, y=group_field, x=numeric_field_id, palette='viridis')
        plt.title(f'Mean {numeric_field_id} by {group_field} (Top 10)')
        plt.xlabel(f'Mean {numeric_field_id}')
        plt.ylabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization: No suitable data available.")

## 6. Conclusion
In this notebook, we have:

- Loaded and inspected the metadata and structure of a detailed proteomics dataset using the Croissant schema and `mlcroissant`.
- Programmatically discovered available record sets and their fields (@id), and dynamically referenced them for further exploration.
- Extracted and analyzed tabular data, performed basic normalization and filtering, and visually explored both distributions and group-level summaries.

This approach can be extended to any Croissant-described dataset, making it straightforward to build reproducible, FAIR pipelines for biomedical and other structured scientific data.

**Next steps:**
- Refine EDA according to your research question
- Adapt filters, normalizations, or groupings to other fields using @id
- Integrate with downstream ML pipelines or knowledge graphs using the structured metadata